# 🖥️ Computer Vision Lab 6-2 — Image Features & Training Mechanics

**BITS F459 — Computer Vision**

Runnable companion to [`labsheet6-2.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-2.md).

**This continues directly from [Lab 6-1](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-1.md)**, reusing the
`dataset.py`, `model.py` and `train.py` you wrote there. Every experiment below is a
**config change**, not new training code.

> ⚠️ **Check your kernel first** — this notebook needs **Python (cv-dl)**, same as 6-1.
>
> Running locally, your Lab 6-1 files are still on disk, so normally you just run the
> check cell below and carry on. The `%%writefile` cells after it are a **fallback** for
> a fresh machine or a cleaned folder — run them only if the check fails.

---
## 0️⃣ Setup

**Step 1 — check you're ready.** This verifies both the kernel and that Lab 6-1's files
are present:

In [ ]:
import sys
print("python:", sys.executable)        # <- must contain 'cv-dl'

try:
    from dataset import build_iris_datasets
    from model import build_model
    from train import train
    print("Lab 6-1 modules imported OK - SKIP the three %%writefile cells below")
except ModuleNotFoundError as exc:
    print(f"{exc} -> RUN the three %%writefile cells below to re-create the files")

**Step 2 — only if the check above failed**, run these three cells to re-create Lab
6-1's files. They are identical to what 6-1 wrote; running them on a folder that already
has your files simply overwrites them with the same content.

In [ ]:
%%writefile dataset.py
"""dataset.py - IRIS"""

import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class IrisDataset(Dataset):
    """A Dataset over in-memory feature/label arrays.

    This is the minimal, complete Dataset: store the data, answer __len__,
    answer __getitem__. Everything else (batching, shuffling) is the
    DataLoader's job, not ours.
    """

    def __init__(self, features, labels):
        # float32 for the features (what nn.Linear expects) and int64 for the
        # labels (what nn.CrossEntropyLoss expects as CLASS INDICES).
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


def build_iris_datasets(val_size=0.2, test_size=0.2, random_state=42):
    """Returns (train_ds, val_ds, test_ds, scaler, class_names, n_features, n_classes).

    Note the three-way split: train to fit weights, val to choose the best
    checkpoint and to trigger early stopping, test touched ONCE at the end.
    """
    data = load_iris()
    X, y = data.data, data.target

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=val_size + test_size, stratify=y, random_state=random_state
    )
    relative_test = test_size / (val_size + test_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_test, stratify=y_temp, random_state=random_state
    )

    # Fit the scaler on TRAIN ONLY, then apply it to val/test - exactly the same
    # rule as Lab 4's "fit the visual vocabulary on training descriptors only".
    # Fitting on all the data leaks val/test statistics into training.
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    return (
        IrisDataset(X_train, y_train),
        IrisDataset(X_val, y_val),
        IrisDataset(X_test, y_test),
        scaler,
        # Plain python str, NOT numpy.str_ - these get written into the
        # checkpoint, and torch.load's weights_only=True default rejects
        # numpy scalar types. See the checkpointing warning in Section 6.
        [str(name) for name in data.target_names],
        X_train.shape[1],   # 4 features
        len(data.target_names),  # 3 classes
    )

In [ ]:
%%writefile model.py
"""model.py"""

import torch.nn as nn


class SingleLayerPerceptron(nn.Module):
    """One nn.Linear, nothing else: logits = W·x + b.

    This is the direct PyTorch equivalent of Lab 4's SGDClassifier - one
    weight matrix, one bias, a linear decision boundary.
    """

    def __init__(self, in_features, num_classes):
        super().__init__()          # ALWAYS call this first in an nn.Module
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # x: (batch_size, in_features)  ->  returns (batch_size, num_classes)
        # Returns RAW LOGITS. No softmax here - see the warning below.
        return self.fc(x)


class MLP(nn.Module):
    """A multi-layer perceptron: one or more hidden layers, each followed by
    a ReLU non-linearity and (optionally) dropout."""

    def __init__(self, in_features, num_classes, hidden_sizes=(128,), dropout=0.0):
        super().__init__()

        layers = []
        prev = in_features
        for hidden in hidden_sizes:
            layers.append(nn.Linear(prev, hidden))
            # Without a non-linearity here, stacked Linear layers collapse
            # algebraically into a single Linear layer (Section 1).
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = hidden

        # Output layer: no activation, no dropout - raw logits.
        layers.append(nn.Linear(prev, num_classes))

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def build_model(name, in_features, num_classes, hidden_sizes=(128,), dropout=0.0):
    if name == "single":
        return SingleLayerPerceptron(in_features, num_classes)
    if name == "mlp":
        return MLP(in_features, num_classes, hidden_sizes, dropout)
    raise ValueError(f"Unknown model: {name}")

In [ ]:
%%writefile train.py
"""train.py - one parameterized training run, logged to TensorBoard and checkpointed."""

import os
import time
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from model import build_model


class OneHotMSELoss(nn.Module):
    """MSE against a one-hot target. Included deliberately as the WRONG tool
    for classification, so you can watch it underperform in Lab 6-2."""

    def __init__(self, num_classes):
        super().__init__()
        self.num_classes = num_classes
        self.mse = nn.MSELoss()

    def forward(self, logits, targets):
        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1, targets.view(-1, 1), 1.0)
        return self.mse(torch.softmax(logits, dim=1), one_hot)


def build_criterion(name, num_classes):
    if name == "cross_entropy":
        return nn.CrossEntropyLoss()      # expects RAW LOGITS + class indices
    if name == "hinge":
        return nn.MultiMarginLoss()       # multiclass hinge == Lab 4's linear SVM
    if name == "mse":
        return OneHotMSELoss(num_classes)
    raise ValueError(f"Unknown loss: {name}")


def build_optimizer(name, params, lr, weight_decay):
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd_momentum":
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    if name == "adam":
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Run the model over a loader and return (mean_loss, accuracy).

    model.eval() matters here: it disables dropout. @torch.no_grad() is a
    separate thing - it stops the graph being recorded (faster, less memory).
    """
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for features, labels in loader:
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, correct / total


def train(config, train_ds, val_ds, test_ds=None, scaler=None, class_names=None):
    """Train one configuration end to end. Returns a summary dict."""

    run_name = config["run_name"]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    log_dir = os.path.join("runs", run_name)
    ckpt_dir = os.path.join("checkpoints", run_name)
    os.makedirs(ckpt_dir, exist_ok=True)
    writer = SummaryWriter(log_dir)

    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False)

    model = build_model(
        config["model"], config["in_features"], config["num_classes"],
        hidden_sizes=config.get("hidden_sizes", (128,)),
        dropout=config.get("dropout", 0.0),
    ).to(device)

    criterion = build_criterion(config["loss"], config["num_classes"])
    optimizer = build_optimizer(
        config["optimizer"], model.parameters(),
        config["lr"], config.get("weight_decay", 0.0),
    )

    # Log the model graph to TensorBoard's GRAPHS tab - a clickable picture of
    # the forward pass you traced by hand in Section 4.
    sample_batch = next(iter(train_loader))[0].to(device)
    writer.add_graph(model, sample_batch)

    best_val_loss = float("inf")
    best_val_acc = 0.0
    best_epoch = -1
    best_state = None
    patience_counter = 0
    start_time = time.perf_counter()

    for epoch in range(1, config["epochs"] + 1):
        # ---------------- train one epoch ----------------
        model.train()                      # dropout ON
        running_loss, correct, total = 0.0, 0, 0

        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            # ---- the five-line ritual (Section 2) ----
            optimizer.zero_grad()
            logits = model(features)
            loss = criterion(logits, labels)

            # Optional L1 regularization. Unlike L2 (which lives in the
            # optimizer as weight_decay), L1 is not built in - you just add
            # the term to the loss, because a loss is only an expression
            # autograd knows how to differentiate.
            l1_lambda = config.get("l1_lambda", 0.0)
            if l1_lambda > 0:
                loss = loss + l1_lambda * sum(p.abs().sum() for p in model.parameters())

            loss.backward()
            optimizer.step()
            # ------------------------------------------

            running_loss += loss.item() * len(labels)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        train_loss = running_loss / total
        train_acc = correct / total
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        # ---------------- TensorBoard ----------------
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
        writer.add_scalar("generalization_gap", train_acc - val_acc, epoch)

        # Weight and GRADIENT histograms. .grad still holds the last batch's
        # gradients at this point - zero_grad() only runs at the top of the
        # next iteration. The gradient histograms are what let you SEE a
        # learning rate explode or a gradient vanish in Lab 6-2.
        for name, param in model.named_parameters():
            writer.add_histogram(f"weights/{name}", param, epoch)
            if param.grad is not None:
                writer.add_histogram(f"grads/{name}", param.grad, epoch)

        # ---------------- checkpointing ----------------
        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            # The optimizer state carries momentum / Adam moment estimates.
            # Resume without it and the loss visibly jumps as they rebuild.
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "config": config,
            "class_names": class_names,
            # Save the fitted scaler WITH the weights - see Section 3's warning.
            "scaler_mean": None if scaler is None else scaler.mean_.tolist(),
            "scaler_scale": None if scaler is None else scaler.scale_.tolist(),
        }
        torch.save(checkpoint, os.path.join(ckpt_dir, "last.pt"))

        # ---------------- best checkpoint + early stopping ----------------
        # NOTE: selection is on VALIDATION loss, never training loss. Val loss
        # and val accuracy can disagree - loss often starts rising from
        # overconfidence while accuracy still creeps up. Pick one, state which.
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc      # accuracy AT the best-loss epoch
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
            torch.save(checkpoint, os.path.join(ckpt_dir, "best.pt"))
            marker = "  <- best"
        else:
            patience_counter += 1
            marker = ""

        print(f"[{run_name}] epoch {epoch:3d}/{config['epochs']}  "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}{marker}")

        if patience_counter >= config.get("patience", config["epochs"]):
            print(f"[{run_name}] early stopping at epoch {epoch} "
                  f"(no val improvement for {patience_counter} epochs). "
                  f"best.pt is from epoch {best_epoch}.")
            break

    elapsed = time.perf_counter() - start_time

    # ---------------- final test evaluation, using the BEST weights ----------------
    # Restore best.pt rather than keeping whatever the last epoch produced -
    # this is the entire reason for tracking a best checkpoint at all.
    test_loss = test_acc = None
    if test_ds is not None and best_state is not None:
        model.load_state_dict(best_state)
        test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        print(f"[{run_name}] TEST (from best.pt, epoch {best_epoch}): "
              f"loss={test_loss:.4f} acc={test_acc:.4f}")

    # hparams tab: makes every run across BOTH halves of Lab 6 comparable
    # in one sortable table
    writer.add_hparams(
        {k: v for k, v in config.items() if isinstance(v, (int, float, str, bool))},
        {"hparam/best_val_loss": best_val_loss,
         "hparam/best_val_acc": best_val_acc,
         "hparam/test_acc": test_acc if test_acc is not None else 0.0},
    )
    writer.close()

    return {
        "run_name": run_name, "best_epoch": best_epoch,
        "best_val_loss": best_val_loss, "best_val_acc": best_val_acc,
        "test_acc": test_acc, "seconds": elapsed,
    }

---
## 1️⃣ Extending `dataset.py`: Converting MNIST into LBP Features

**There is no "LBP dataset".** MNIST ships raw 28×28 images; `MNISTLBPDataset` is a
**converter**.

```
28×28 uint8 image
   └─> local_binary_pattern(P=8, R=1, "uniform")  ->  28×28 code map, values 0..9
        └─> split into a 4×4 grid of 7×7 cells
             └─> bincount each cell into 10 bins, L1-normalize
                  └─> concatenate  ->  4 × 4 × 10 = 160-d feature vector
```

A histogram of the *whole* map would be 10 numbers for a 10-class problem, and would
discard **where** each pattern occurred — which for handwritten digits is most of the
information. The grid fixes that.

> 🔎 This transform is **stateless** — nothing is learned from data — so unlike Lab 4's
> BoVW vocabulary there is nothing to leak.

> ⚠️ **MNIST downloads fail often.** `yann.lecun.com` now 404s permanently and
> torchvision's S3 mirror is flaky, so `load_mnist_arrays()` falls back to OpenML.

The next cell **appends** to `dataset.py` — `IrisDataset` stays as it is.

In [ ]:
%%writefile -a dataset.py


"""dataset.py (continued) - MNIST, as LBP features and as raw pixels.

Append this to the dataset.py you wrote in Lab 6-1.
"""

import os
import numpy as np
import torch
from torch.utils.data import Dataset
from torchvision.datasets import MNIST
from skimage.feature import local_binary_pattern

LBP_P = 8        # number of circular neighbours

LBP_R = 1        # radius, in pixels
LBP_BINS = 10    # uniform LBP with P=8 produces P+2 = 10 distinct codes
LBP_GRID = 4     # 4x4 grid of cells -> 4*4*10 = 160-d feature


def load_mnist_arrays(root="./data", train=True):
    """Returns (images, targets) as a (N, 28, 28) uint8 array and an int64 tensor.

    Primary path is torchvision, whose MNIST exposes .data and .targets
    directly - so we skip PIL and ToTensor() and hand numpy straight to skimage.

    Fallback: torchvision's MNIST mirrors go down regularly, and the original
    yann.lecun.com URL now returns 404 permanently. Rather than leaving you
    stuck, we pull the identical data from OpenML if the download fails.
    """
    try:
        base = MNIST(root=root, train=train, download=True)
        return base.data.numpy(), base.targets.clone()
    except Exception as exc:
        print(f"[mnist] torchvision download failed ({type(exc).__name__}); "
              f"falling back to OpenML...")
        from sklearn.datasets import fetch_openml
        X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                            as_frame=False, parser="auto")
        X = X.astype(np.uint8).reshape(-1, 28, 28)
        y = y.astype(np.int64)
        # OpenML ships one combined set; the canonical split is first 60k / last 10k.
        span = slice(0, 60000) if train else slice(60000, 70000)
        return X[span], torch.tensor(y[span])


def lbp_feature(image_u8, P=LBP_P, R=LBP_R, grid=LBP_GRID, n_bins=LBP_BINS):
    """One 28x28 uint8 image -> a (grid*grid*n_bins,)-d float32 feature vector.

    Stateless: no fitting, no parameters learned from data. Train, val and
    test all go through this identical function.
    """
    codes = local_binary_pattern(image_u8, P, R, method="uniform")

    h, w = codes.shape
    row_edges = np.linspace(0, h, grid + 1).astype(int)
    col_edges = np.linspace(0, w, grid + 1).astype(int)

    blocks = []
    for i in range(grid):
        for j in range(grid):
            cell = codes[row_edges[i]:row_edges[i + 1],
                         col_edges[j]:col_edges[j + 1]].astype(np.int64).ravel()
            # bincount is much faster than np.histogram for small integer codes
            hist = np.bincount(cell, minlength=n_bins)[:n_bins].astype(np.float32)
            total = hist.sum()
            if total > 0:
                hist /= total   # L1-normalize: a distribution, not a raw count
            blocks.append(hist)

    return np.concatenate(blocks)


class MNISTLBPDataset(Dataset):
    """MNIST converted to grid-LBP feature vectors.

    The conversion is done ONCE in __init__ and cached to disk, not recomputed
    in __getitem__ - see the note below on why that matters.
    """

    def __init__(self, root="./data", train=True, cache_dir="./cache",
                 limit=None, random_state=42):
        split = "train" if train else "test"
        os.makedirs(cache_dir, exist_ok=True)
        cache_path = os.path.join(
            cache_dir, f"mnist_lbp_{split}_P{LBP_P}_R{LBP_R}_g{LBP_GRID}.pt"
        )

        if os.path.exists(cache_path):
            blob = torch.load(cache_path)
            features, targets = blob["features"], blob["targets"]
            print(f"[MNISTLBPDataset] loaded cached {split} features from {cache_path}")
        else:
            images, targets = load_mnist_arrays(root=root, train=train)

            print(f"[MNISTLBPDataset] converting {len(images)} {split} images to LBP "
                  f"features (one-time, then cached)...")
            features = torch.tensor(
                np.stack([lbp_feature(im) for im in images]), dtype=torch.float32
            )
            torch.save({"features": features, "targets": targets}, cache_path)
            print(f"[MNISTLBPDataset] cached to {cache_path}")

        # Subsample AFTER caching, so changing `limit` never triggers a reconversion.
        if limit is not None and limit < len(targets):
            generator = torch.Generator().manual_seed(random_state)
            keep = torch.randperm(len(targets), generator=generator)[:limit]
            features, targets = features[keep], targets[keep]

        self.features = features
        self.targets = targets
        self.n_features = features.shape[1]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]


class MNISTRawDataset(Dataset):
    """MNIST as flattened, scaled raw pixels (784-d).

    This is the CONTROL for the LBP experiment - without it, an LBP accuracy
    number has nothing to be compared against and means very little.
    """

    def __init__(self, root="./data", train=True, limit=None, random_state=42):
        images, targets = load_mnist_arrays(root=root, train=train)
        features = torch.tensor(
            images.reshape(len(images), -1).astype(np.float32) / 255.0
        )

        if limit is not None and limit < len(targets):
            generator = torch.Generator().manual_seed(random_state)
            keep = torch.randperm(len(targets), generator=generator)[:limit]
            features, targets = features[keep], targets[keep]

        self.features = features
        self.targets = targets
        self.n_features = features.shape[1]   # 784

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

> ⏱️ **Why convert in `__init__`, not `__getitem__`?** `__getitem__` runs once per
> sample per epoch, so a 30-epoch run would recompute identical features 30 times and
> starve the GPU. Converting once and caching means you pay once, ever. The conversion
> runs at ~0.2 ms/image (tens of seconds for 60k, one time); cached reloads are a
> fraction of a second.

---
## 2️⃣ Start TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

### Running It: LBP Features vs Raw Pixels

Same `train()`, same models — only the **representation** changes. The first
`MNISTLBPDataset(...)` call downloads MNIST and runs the one-time conversion.

In [ ]:
"""run_lab6_mnist.py"""

import torch
from torch.utils.data import random_split

from dataset import MNISTLBPDataset, MNISTRawDataset
from train import train

# The first MNISTLBPDataset construction downloads MNIST and runs the one-time
# LBP conversion; every later run loads the cached tensor instantly.
lbp_train = MNISTLBPDataset(train=True)
lbp_test = MNISTLBPDataset(train=False)
# Carve a validation split out of the training data - MNIST ships only train/test.
lbp_val, lbp_train = random_split(
    lbp_train, [6000, len(lbp_train) - 6000],
    generator=torch.Generator().manual_seed(42),
)

base_mnist = dict(
    num_classes=10, batch_size=128, epochs=30,
    loss="cross_entropy", optimizer="adam", lr=1e-3, weight_decay=0.0,
    patience=5,
)

train({**base_mnist, "run_name": "mnist_lbp_single", "model": "single",
       "in_features": 160}, lbp_train, lbp_val, lbp_test)

train({**base_mnist, "run_name": "mnist_lbp_mlp", "model": "mlp",
       "in_features": 160, "hidden_sizes": (128,)}, lbp_train, lbp_val, lbp_test)

# The control: identical everything, raw 784-d pixels instead of 160-d LBP.
raw_train = MNISTRawDataset(train=True)
raw_test = MNISTRawDataset(train=False)
raw_val, raw_train = random_split(
    raw_train, [6000, len(raw_train) - 6000],
    generator=torch.Generator().manual_seed(42),
)

train({**base_mnist, "run_name": "mnist_raw_single", "model": "single",
       "in_features": 784}, raw_train, raw_val, raw_test)

train({**base_mnist, "run_name": "mnist_raw_mlp", "model": "mlp",
       "in_features": 784, "hidden_sizes": (128,)}, raw_train, raw_val, raw_test)

### 📝 Task

1. Report best epoch, best val loss and test accuracy for all four runs.
2. **Single vs multi layer**, on each representation.
3. **LBP vs raw pixels** — which wins, by how much, at what dimension? LBP is 160-d
   against raw pixels' 784-d, so "which is better" and "which is better *per dimension*"
   are two different questions. Answer both.
4. Think about what each encodes. LBP describes *local texture patterns* binned by
   region; raw pixels keep everything including position but know nothing about
   structure. MNIST digits are near-binary strokes on flat background — how much genuine
   *texture* is there, and how much of LBP's performance is really the 4×4 grid
   preserving *where* things are? **Report what you measure — don't assume the
   hand-designed feature has to lose.**

---
## 3️⃣ Training Mechanics

### 3.0 The overfitting sandbox

> ⚠️ On full MNIST a small MLP barely overfits — curves stay glued together and weight
> decay and dropout change nothing visible. To study overfitting you need a setup that
> actually overfits: **subsample training to ~1000 images.**

In [ ]:
sandbox_train = MNISTLBPDataset(train=True, limit=1000)
sandbox_val = MNISTLBPDataset(train=False, limit=2000)

sandbox = dict(
    in_features=160, num_classes=10, model="mlp", hidden_sizes=(256, 256),
    batch_size=32, epochs=60, loss="cross_entropy", optimizer="adam",
    lr=1e-3, patience=60,   # patience disabled, so you can SEE the overfit happen
)

### 3.1 Loss function

| Loss | PyTorch | Notes |
|---|---|---|
| Cross-entropy | `nn.CrossEntropyLoss()` | The default. Raw logits + class indices. |
| Multiclass hinge | `nn.MultiMarginLoss()` | This *is* Lab 4's linear SVM objective. |
| MSE on one-hot | `OneHotMSELoss` | The wrong tool, to see *how* it's wrong. |

> ⚠️ **Compare on *accuracy*, never on loss.** The three produce numbers on completely
> different scales. The MSE run may well report the **lowest loss while being the worst
> classifier** — which is why `train()` returns `best_val_acc` too.

In [ ]:
results = {}
for loss_name in ["cross_entropy", "hinge", "mse"]:
    results[loss_name] = train({**sandbox, "run_name": f"loss_{loss_name}", "loss": loss_name},
                               sandbox_train, sandbox_val)

print(f"\n{'loss':16s} {'val_acc':>9s} {'val_loss':>10s}")
for k, r in results.items():
    print(f"{k:16s} {r['best_val_acc']:9.4f} {r['best_val_loss']:10.4f}   <- loss NOT comparable")

### 3.2 Optimizer

Compare **epochs to reach a target accuracy** and **curve smoothness**, not just the
final number. The last two runs make the real point: **optimizer and learning rate are
not independent knobs.**

In [ ]:
for opt_name, lr in [("sgd", 0.1), ("sgd_momentum", 0.1), ("adam", 1e-3), ("adamw", 1e-3)]:
    r = train({**sandbox, "run_name": f"opt_{opt_name}", "optimizer": opt_name, "lr": lr},
              sandbox_train, sandbox_val)
    print(f"{opt_name:14s} lr={lr:<8} val_acc={r['best_val_acc']:.4f}")

for opt_name, lr in [("adam", 0.1), ("sgd", 1e-3)]:
    r = train({**sandbox, "run_name": f"mismatch_{opt_name}_{lr}", "optimizer": opt_name, "lr": lr},
              sandbox_train, sandbox_val)
    print(f"MISMATCH {opt_name:8s} lr={lr:<8} val_acc={r['best_val_acc']:.4f}")

### 3.3 Learning rate

| `lr` | Expected |
|---|---|
| `1e-5` | Loss barely moves — learning, unusably slowly |
| `1e-3` | Healthy convergence (Adam) |
| `1.0` | Collapses to ~chance (0.1 for 10 classes), or goes `NaN` |

Afterwards open **HISTOGRAMS** and compare `grads/*` across the three.

> 🔎 A collapsed run and a NaN run are both failures, but the collapsed one is more
> dangerous: no error, no crash, and a loss number that still prints.

In [ ]:
for lr in [1e-5, 1e-3, 1.0]:
    r = train({**sandbox, "run_name": f"lr_{lr}", "lr": lr}, sandbox_train, sandbox_val)
    print(f"lr={lr:<10} val_acc={r['best_val_acc']:.4f}  val_loss={r['best_val_loss']:.4f}")

### 3.4 Regularization

In PyTorch **L2 is an optimizer argument**, not a loss term — same thing
mathematically, since the derivative of `λ‖w‖²` is `2λw`. **L1 is not built in**, which
is why `train()` adds it by hand; that block shows a loss is just an expression autograd
can differentiate.

Watch `generalization_gap` in TensorBoard: it should narrow, then at the high end the
model underfits and *both* curves worsen.

In [ ]:
for wd in [0.0, 1e-4, 1e-2, 1e-1]:
    r = train({**sandbox, "run_name": f"wd_{wd}", "weight_decay": wd}, sandbox_train, sandbox_val)
    print(f"weight_decay={wd:<8} val_acc={r['best_val_acc']:.4f}")

r = train({**sandbox, "run_name": "l1_1e-5", "l1_lambda": 1e-5}, sandbox_train, sandbox_val)
print(f"l1_lambda=1e-5   val_acc={r['best_val_acc']:.4f}")

### 3.5 Dropout — and why `model.eval()` is not optional

Dropout needs a hidden layer, so it is inherently an **MLP** technique.

In [ ]:
for p in [0.0, 0.2, 0.5]:
    r = train({**sandbox, "run_name": f"dropout_{p}", "dropout": p}, sandbox_train, sandbox_val)
    print(f"dropout={p:<6} val_acc={r['best_val_acc']:.4f}")

Now the most valuable experiment here — same model, same data, evaluated in each mode:

In [ ]:
"""eval_mode_demo.py - why model.eval() is not optional."""
import torch
from torch.utils.data import DataLoader
from dataset import MNISTLBPDataset
from model import build_model

# NOTE: we deliberately do NOT reuse train.py's evaluate() here, because it
# calls model.eval() internally - which would silently fix the very bug we
# are trying to demonstrate. This helper leaves the mode exactly as set.
@torch.no_grad()
def accuracy_leaving_mode_alone(model, loader):
    correct = total = 0
    for features, labels in loader:
        correct += (model(features).argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return correct / total


train_ds = MNISTLBPDataset(train=True, limit=1000)
val_ds = MNISTLBPDataset(train=False, limit=2000)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)

model = build_model("mlp", 160, 10, hidden_sizes=(256, 256), dropout=0.5)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for _ in range(10):
    model.train()
    for features, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()

model.train()   # BUG: dropout is STILL ACTIVE during evaluation
print("train() mode:", [round(accuracy_leaving_mode_alone(model, val_loader), 4)
                        for _ in range(5)])

model.eval()    # correct: dropout disabled
print("eval()  mode:", [round(accuracy_leaving_mode_alone(model, val_loader), 4)
                        for _ in range(5)])

In `train()` mode accuracy is **lower and different every time** — you're randomly
deleting half the network each forward pass. In `eval()` mode it's higher and identical.

> ⚠️ **Note the trap this demo had to avoid.** `train.py`'s `evaluate()` starts with
> `model.eval()`. Call *that* to run this experiment and both halves print the same
> numbers — the helper quietly repairs the bug. Any time you try to reproduce a bug and
> it refuses to appear, check whether something downstream is fixing it for you.

### 📝 Task — the experiment matrix

Present the runs above as **one table plus overlaid TensorBoard screenshots**, and for
each group state **which setting you would ship and why**, in one sentence.

Then re-enable early stopping (`patience=5`) on your best config and report **which
epoch `best.pt` came from versus which epoch training stopped** — that gap is what early
stopping and best-checkpointing bought you.

---
## 4️⃣ Inference: Loading `best.pt`

> 💡 This uses **matplotlib** rather than `cv2.imshow()`. Inside a Jupyter notebook,
> `cv2.imshow()` opens a separate OS window that frequently hangs the kernel — matplotlib
> renders inline instead. The labsheet's standalone `predict.py` does use `cv2.imshow()`,
> which is fine when run as a plain script from the terminal.

In [ ]:
"""Classify one input using a saved best.pt."""
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from model import build_model
from dataset import lbp_feature, load_mnist_arrays

CHECKPOINT_PATH = "checkpoints/mnist_lbp_mlp/best.pt"

images, targets = load_mnist_arrays(train=False)
cv2.imwrite("my_digit.png", images[0])
print("test digit written, true label =", int(targets[0]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
config = ckpt["config"]
print(f"Loaded checkpoint from epoch {ckpt['epoch']} "
      f"(val_loss={ckpt['val_loss']:.4f}, val_acc={ckpt['val_acc']:.4f})")

# Rebuild the SAME architecture the checkpoint was trained with - this is why
# we saved `config` alongside the state_dict.
model = build_model(
    config["model"], config["in_features"], config["num_classes"],
    hidden_sizes=tuple(config.get("hidden_sizes", (128,))),
    dropout=config.get("dropout", 0.0),
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()          # disables dropout - NOT optional

gray = cv2.imread("my_digit.png", cv2.IMREAD_GRAYSCALE)
gray = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)
features = torch.tensor(lbp_feature(gray), dtype=torch.float32).unsqueeze(0).to(device)

if ckpt.get("scaler_mean") is not None:
    mean = torch.tensor(ckpt["scaler_mean"], dtype=torch.float32, device=device)
    scale = torch.tensor(ckpt["scaler_scale"], dtype=torch.float32, device=device)
    features = (features - mean) / scale

with torch.no_grad():
    logits = model(features)
    # Softmax belongs HERE, at display time - not in forward().
    probabilities = torch.softmax(logits, dim=1)[0]

predicted = int(probabilities.argmax())
confidence = float(probabilities[predicted])
class_names = ckpt.get("class_names") or [str(i) for i in range(config["num_classes"])]
print(f"Predicted: {class_names[predicted]}  (confidence {confidence:.3f})")

plt.figure(figsize=(3, 3))
plt.imshow(gray, cmap="gray")
plt.title(f"{class_names[predicted]}  ({confidence:.2f})")
plt.axis("off")
plt.show()

---
## 💾 What you produced

Everything is already on disk in this folder — `runs/` for TensorBoard, `checkpoints/`
for the weights, `cache/` for the precomputed LBP features (so re-running never repeats
the conversion). Nothing to download.

In [ ]:
import os

for folder in ["runs", "checkpoints"]:
    if os.path.isdir(folder):
        entries = sorted(os.listdir(folder))
        print(f"{folder}/  ({len(entries)} runs)")
        for sub in entries:
            print("  ", sub)
        print()

if os.path.isdir("cache"):
    print("cache/ ", sorted(os.listdir("cache")))

---
## 🧪 Student Task (Evaluative) — Feature & Architecture Study

Full description in [`labsheet6-2.md`](https://github.com/Aritra/Computer_Vision/blob/main/labsheets/labsheet6-2.md).

Train every combination of:

- **Representation:** raw pixels (784-d), LBP `grid=1` (10-d), `grid=2` (40-d),
  `grid=4` (160-d), `grid=7` (490-d)
- **Model:** single-layer, MLP with one hidden layer, MLP with two hidden layers

15 runs. Each is a config change — if you're copy-pasting a training loop, reread Lab
6-1 Section 5.

Report representation, feature dimension, model, best epoch, test accuracy, training
time, and **one-time feature extraction time** (separately — a representation that
trains fast but takes ten minutes to encode is not fast). **Tune the learning rate per
representation**; a 10-d and a 784-d input don't want the same one.

*Think about:* `grid=1` gives a **10-dimensional** feature for a **10-class** problem —
predict what happens before you run it. And `grid=1`→`grid=7` adds spatial information
*and* dimensionality at once; design a comparison that separates them.

In [ ]:
# Starter: build the grid variants for the student task.
# lbp_feature() takes a `grid` argument, so you can parameterize MNISTLBPDataset
# or map lbp_feature over the raw images yourself. The cache filename encodes the
# grid size, so variants won't collide.

from dataset import lbp_feature, load_mnist_arrays

images, targets = load_mnist_arrays(train=False)
for g in [1, 2, 4, 7]:
    print(f"grid={g}  ->  {len(lbp_feature(images[0], grid=g))}-d feature")